In [4]:
#!/usr/bin/env python3
"""
Spam Email Classifier
=====================

Trains and compares two classic text classifiers on email content:

  * Multinomial Naive Bayes  - fast, probabilistic, the traditional spam filter
  * Linear SVM (LinearSVC)   - usually the stronger of the two on TF-IDF features

Pipeline:  raw email -> normalise (URLs, money, numbers, SHOUTING) -> TF-IDF
           (1-2 word n-grams) -> classifier, tuned with 5-fold cross-validated
           grid search, then scored on a held-out test set.

Usage
-----
  python spam_classifier.py                           # train on built-in sample data, compare, save best
  python spam_classifier.py --data spam.csv           # train on your own data (see below)
  python spam_classifier.py --predict "Subject: hi\\n\\nYou WON $1000, click http://x.top"
  python spam_classifier.py --predict-file mail1.txt mail2.txt
  python spam_classifier.py --save-sample data/       # write the sample data to data/emails.csv

Accepted --data formats
  * CSV with a label column (label / v1 / class / category) and a text column
    (text / v2 / message / email / body). Labels: spam/ham or 1/0.
    This covers the Kaggle "spam.csv" (SMS Spam Collection) as-is.
  * Headerless tab-separated "ham<TAB>message" lines (original SMS Spam Collection file).
"""
from __future__ import annotations

import argparse
import random
import re
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


# --------------------------------------------------------------------------- #
# 1. Text normalisation
# --------------------------------------------------------------------------- #
_URL = re.compile(r"(https?://\S+|www\.\S+)", re.I)
_EMAIL = re.compile(r"\S+@\S+\.\S+")
_MONEY = re.compile(r"[$£€₹]\s?\d[\d,\.]*|\b\d[\d,\.]*\s?(?:usd|dollars|rupees|inr)\b", re.I)
_NUM = re.compile(r"\b\d+\b")
_CAPS = re.compile(r"\b[A-Z]{3,}\b")


def normalize(text: str) -> str:
    """Collapse noisy specifics into shared tokens and keep a few 'style' signals.

    Every distinct URL or dollar amount is unique, so on their own they would
    never repeat across emails. Replacing them with a token lets the model learn
    "emails containing a URL and a money amount look like spam" instead.
    Lower-casing would hide SHOUTING, so we record it as an extra token first.
    """
    words = max(len(text.split()), 1)
    caps = len(_CAPS.findall(text))
    out = _URL.sub(" urltoken ", text)
    out = _EMAIL.sub(" emailtoken ", out)
    out = _MONEY.sub(" moneytoken ", out).lower()
    out = _NUM.sub(" numtoken ", out)
    extras = []
    if caps >= 3 or caps / words > 0.1:
        extras.append("manycapstoken")
    if text.count("!") >= 2:
        extras.append("manyexclaimtoken")
    return out + " " + " ".join(extras)


def make_pipeline(clf) -> Pipeline:
    return Pipeline([
        ("tfidf", TfidfVectorizer(preprocessor=normalize, ngram_range=(1, 2), min_df=2,
                                  sublinear_tf=True, stop_words="english")),
        ("clf", clf),
    ])


# --------------------------------------------------------------------------- #
# 2. Data: built-in synthetic sample + loader for real datasets
# --------------------------------------------------------------------------- #
NAMES = ["Alex", "Priya", "Sam", "Maria", "Chen", "Omar", "Nina", "Ravi", "Jordan", "Aisha", "Tom", "Lena"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "next week", "tomorrow"]
TIMES = ["9am", "10:30am", "2pm", "3:30pm", "4pm"]
TOPICS = ["budget", "design review", "sprint planning", "hiring", "onboarding", "quarterly planning", "vendor"]
PROJECTS = ["the website redesign", "the Q3 report", "the mobile app", "the data migration", "the client proposal"]
CUISINES = ["Thai", "Italian", "Mexican", "Korean", "Indian"]
BANKS = ["Chase", "HSBC", "Wells Fargo", "PayPal", "Citibank", "SBI"]
MONTHS = ["January", "March", "June", "August", "October", "November"]
SYSTEMS = ["staging server", "payment API", "VPN", "database", "build pipeline"]


def _url(r: random.Random, spam: bool) -> str:
    if spam and r.random() < 0.85:
        return (f"http://{r.choice(['win', 'cheap', 'secure', 'verify', 'bonus', 'cash'])}-"
                f"{r.choice(['prize', 'meds', 'bank', 'offer', 'now', 'deal'])}{r.randint(1, 99)}."
                f"{r.choice(['biz', 'xyz', 'info', 'top', 'net'])}/{r.choice(['claim', 'login', 'buy', 'apply'])}")
    return (f"https://{r.choice(['docs', 'meet', 'track', 'mail'])}."
            f"{r.choice(['company', 'example', 'shopfast', 'acme'])}.com/{r.randint(100, 9999)}")


def _ham(r: random.Random) -> tuple[str, str]:
    n, n2, d, t = r.choice(NAMES), r.choice(NAMES), r.choice(DAYS), r.choice(TIMES)
    kind = r.randrange(10)
    if kind == 0:
        topic = r.choice(TOPICS)
        return (f"{r.choice(['', 'Re: ', 'Fwd: '])}{topic} meeting {d}",
                f"Hi {n}, can we move the {topic} meeting to {d} at {t}? I've updated the agenda, "
                f"let me know if that works for you. Thanks, {n2}")
    if kind == 1:
        p = r.choice(PROJECTS)
        return (f"Update on {p}",
                f"Hi {n}, quick update on {p}: the first draft is done but the review is still pending. "
                f"Could you take a look before {d}? Happy to walk you through it. Best, {n2}")
    if kind == 2:
        return ("Dinner?",
                f"Hey {n}! Are you free for dinner on {d}? Thinking of trying that new {r.choice(CUISINES)} "
                f"place downtown. Let me know and I'll book a table. - {n2}")
    if kind == 3:  # legit mail that shares words with spam (order, free, click)
        return (f"Your order #{r.randint(10000, 99999)} has shipped",
                f"Hi {n}, good news: your order has shipped and should arrive by {d}. Track your package "
                f"here: {_url(r, False)}. Free returns within 30 days. Thank you for shopping with us.")
    if kind == 4:
        return (f"{r.choice(TOPICS).title()} weekly digest",
                f"This week: five articles on {r.choice(TOPICS)} and {r.choice(TOPICS)}. As a subscriber you "
                f"get a special offer of 10% off the annual plan. Read online: {_url(r, False)}. "
                f"You can unsubscribe at any time from your account settings.")
    if kind == 5:  # legit 'urgent' mail
        return (f"URGENT: {r.choice(SYSTEMS)} is down",
                f"Team, the {r.choice(SYSTEMS)} went down at {t}. Please join the call now: {_url(r, False)}. "
                f"I'm free to talk through the fix and will post updates in the channel. {n2}")
    if kind == 6:  # legit bank mail (overlaps with phishing vocabulary)
        return (f"Your {r.choice(MONTHS)} statement is ready",
                f"Hello {n}, your {r.choice(MONTHS)} statement is now available. Your account balance is "
                f"${r.randint(200, 9000):,}. View it in the {r.choice(BANKS)} app or at {_url(r, False)}. "
                f"We will never ask for your password by email.")
    if kind == 7:
        return (f"Assignment {r.randint(1, 9)} due {d}",
                f"Hi class, a reminder that assignment {r.randint(1, 9)} is due {d}. Please submit it through "
                f"the course portal. Office hours are on {r.choice(DAYS)} at {t} if you have questions.")
    if kind == 8:
        return ("Team lunch on Friday",
                f"Hi all, the company is buying free lunch for the team on Friday at noon. Please RSVP by "
                f"{d} so we order enough food. Dietary restrictions? Tell {n}.")
    # legit invoices: same vocabulary as the fake-invoice spam (invoice, payment, due), different context
    return (f"Invoice for {r.choice(MONTHS)}",
            f"Hi {n}, please find the {r.choice(MONTHS)} invoice for {r.choice(PROJECTS)} attached. "
            f"Net 30 terms apply, so payment is due next month. Ping me if anything looks off. {n2}")


def _style(r: random.Random, subj: str, body: str) -> tuple[str, str]:
    if r.random() < 0.35:
        subj = subj.upper()
    if r.random() < 0.4:
        body += r.choice(["!!!", "!!", " !"])
    return subj, body


def _spam(r: random.Random) -> tuple[str, str]:
    n, d = r.choice(NAMES), r.choice(DAYS)
    u = _url(r, True)
    amt = f"${r.choice([500, 1000, 2500, 10000, 50000, 1000000]):,}"
    pct = r.choice([50, 70, 80, 90])
    kind = r.randrange(8)
    if kind == 0:
        s, b = (r.choice(["You have won!", "Congratulations, winner", "Claim your prize", "Your free gift is waiting"]),
                f"{r.choice(['Congratulations', 'Dear winner', 'Hello lucky user'])} {n}! You have been selected "
                f"to win {amt} in our {r.choice(['lottery', 'sweepstakes', 'prize draw'])}. "
                f"{r.choice(['Claim your free prize now', 'Click here to claim', 'Reply with your details to claim'])}: "
                f"{u} {r.choice(['Offer expires today', 'Act fast, limited time', 'Hurry, only a few prizes left'])}")
    elif kind == 1:
        s, b = (r.choice(["Cheap meds online", "No prescription needed", f"{pct}% discount on pills"]),
                f"Buy cheap {r.choice(['pills', 'meds', 'tablets', 'supplements'])} online. No prescription needed. "
                f"Discount up to {pct}% off. Discreet shipping. Order now: {u}")
    elif kind == 2:
        s, b = (r.choice(["Instant loan approval", "Bad credit? No problem", f"Get {amt} today"]),
                f"Get approved for a {amt} loan today. Bad credit OK, no credit check, instant cash in your "
                f"account within hours. Apply now at {u} Limited offer")
    elif kind == 3:
        bank = r.choice(BANKS)
        s, b = (r.choice([f"Your {bank} account is suspended", "Security alert", "Action required: verify now"]),
                f"Dear customer, your {bank} account has been suspended due to unusual activity. Verify your "
                f"account immediately to restore access: {u} Failure to respond within 24 hours will result "
                f"in permanent closure.")
    elif kind == 4:
        coin = r.choice(["Bitcoin", "crypto", "Ethereum"])
        s, b = (r.choice([f"Double your {coin} in 7 days", "Guaranteed returns", "Exclusive investment group"]),
                f"Double your {coin} in 7 days! Guaranteed returns of {pct * 5}%. Only a few spots left in our "
                f"exclusive investment group. Join today: {u}")
    elif kind == 5:
        s, b = (r.choice(["Work from home", f"Earn {amt} per week", "Be your own boss"]),
                f"Earn {amt} per month working from home. No experience needed, be your own boss, set your "
                f"own hours. Click here to start earning now: {u}")
    elif kind == 6:  # plain, lower-case 'business opportunity' spam: no shouting to give it away
        return (f"business opportunity for {n.lower()}",
                f"hi {n.lower()}, i found your email and wanted to share a business opportunity. we sell "
                f"{r.choice(['watches', 'handbags', 'sunglasses', 'phone cases'])} at wholesale prices, "
                f"low minimum order, free sample. reply for the price list or visit {u}")
    else:  # fake invoice
        return (f"Invoice #{r.randint(1000, 9999)} overdue",
                f"Invoice attached. Payment is overdue. Please open the attachment and pay {amt} to avoid "
                f"legal action. Details: {u}")
    return _style(r, s, b)


def build_sample_data(n: int = 2000, spam_frac: float = 0.3, seed: int = 7,
                      label_noise: float = 0.02) -> pd.DataFrame:
    """Synthetic emails. ~2% of labels are flipped, as real datasets contain mistakes."""
    r = random.Random(seed)
    rows = []
    for _ in range(n):
        is_spam = r.random() < spam_frac
        subj, body = (_spam if is_spam else _ham)(r)
        label = int(is_spam)
        if r.random() < label_noise:
            label = 1 - label
        rows.append((label, f"Subject: {subj}\n\n{body}"))
    return pd.DataFrame(rows, columns=["label", "text"])


_LABEL_COLS = ("label", "v1", "class", "category", "type")
_TEXT_COLS = ("text", "v2", "message", "email", "body", "content", "sms")


def load_dataset(path: str) -> pd.DataFrame:
    p = Path(path)
    with open(p, encoding="latin-1") as f:
        first = f.readline()
    if re.match(r"^(ham|spam)\t", first, re.I):  # original SMS Spam Collection
        df = pd.read_csv(p, sep="\t", header=None, names=["label", "text"], encoding="latin-1", quoting=3)
    else:
        df = pd.read_csv(p, encoding="latin-1")
        cols = {c.lower().strip(): c for c in df.columns}
        lab = next((cols[c] for c in _LABEL_COLS if c in cols), None)
        txt = next((cols[c] for c in _TEXT_COLS if c in cols), None)
        if lab is None or txt is None:
            raise ValueError(f"could not find label/text columns in {list(df.columns)}")
        df = df[[lab, txt]].rename(columns={lab: "label", txt: "text"})
    df = df.dropna().copy()
    mapped = df["label"].astype(str).str.strip().str.lower().map(
        {"spam": 1, "ham": 0, "1": 1, "0": 0, "1.0": 1, "0.0": 0, "true": 1, "false": 0})
    if mapped.isna().any():
        raise ValueError("labels must be spam/ham or 1/0")
    df["label"].update(mapped.dropna().astype(int))
    return df.drop_duplicates("text").reset_index(drop=True)


# --------------------------------------------------------------------------- #
# 3. Training, comparison, inspection
# --------------------------------------------------------------------------- #
SEARCH = {
    "Naive Bayes": (MultinomialNB(), {"clf__alpha": [0.01, 0.03, 0.1, 0.3, 1.0]}),
    "Linear SVM": (LinearSVC(class_weight="balanced", random_state=0), {"clf__C": [0.1, 0.3, 1, 3, 10]}),
}


def train_and_compare(df: pd.DataFrame, test_size: float = 0.2, seed: int = 0):
    X_tr, X_te, y_tr, y_te = train_test_split(df["text"], df["label"], test_size=test_size,
                                              stratify=df["label"], random_state=seed)
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    rows, searches = [], {}
    for name, (model, grid) in SEARCH.items():
        gs = GridSearchCV(make_pipeline(model), grid, scoring="f1", cv=cv).fit(X_tr, y_tr)
        pred = gs.predict(X_te)
        tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
        rows.append({
            "model": name,
            "best params": {k.split("__")[1]: v for k, v in gs.best_params_.items()},
            "cv F1": gs.best_score_,
            "accuracy": accuracy_score(y_te, pred),
            "precision": precision_score(y_te, pred),
            "recall": recall_score(y_te, pred),
            "F1": f1_score(y_te, pred),
            "ham->spam (FP)": int(fp),
            "spam missed (FN)": int(fn),
        })
        searches[name] = gs
    res = pd.DataFrame(rows).set_index("model")
    return res, searches, (len(X_tr), len(X_te))


def top_terms(pipe: Pipeline, k: int = 8) -> tuple[list[str], list[str]]:
    """Most spam-indicative and most ham-indicative terms the model learned."""
    names = pipe.named_steps["tfidf"].get_feature_names_out()
    clf = pipe.named_steps["clf"]
    w = (clf.feature_log_prob_[1] - clf.feature_log_prob_[0]) if hasattr(clf, "feature_log_prob_") else clf.coef_[0]
    order = np.argsort(w)
    return list(names[order[::-1][:k]]), list(names[order[:k]])


def classify(model: Pipeline, texts: list[str]) -> pd.DataFrame:
    """Naive Bayes score = P(spam). SVM score = signed distance from the boundary (>0 means spam)."""
    pred = model.predict(texts)
    score = model.predict_proba(texts)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(texts)
    preview = [re.sub(r"\s+", " ", t)[:70] for t in texts]
    return pd.DataFrame({"verdict": np.where(pred == 1, "SPAM", "ham"), "score": np.round(score, 3),
                         "email": preview})


DEMO_EMAILS = [
    "Subject: Lunch tomorrow?\n\nHey Sam, are you free for lunch tomorrow at 1pm? Thinking of the new place near the office.",
    "Subject: WINNER!! Claim your FREE gift card\n\nYou have been selected to receive a $500 gift card. Click http://claim-gift77.top/now immediately, offer expires today!!!",
    "Subject: Your account has been suspended\n\nDear customer, we detected unusual activity. Verify your account within 24 hours at http://secure-bank12.xyz/login or it will be permanently closed.",
    "Subject: Re: invoice for September\n\nHi Priya, attaching the September invoice for the design work. Payment is due by the 30th, no rush. Thanks!",
    "Subject: Free pizza Friday\n\nTeam, free pizza in the kitchen on Friday at noon. Please RSVP by Thursday so we order enough.",
    "Subject: Cheap meds, no prescription\n\nBuy cheap pills online at 80% discount. No prescription needed. Order now.",
]


# --------------------------------------------------------------------------- #
# 4. CLI
# --------------------------------------------------------------------------- #
def main(argv=None):
    ap = argparse.ArgumentParser(description="Spam email classifier (Naive Bayes vs Linear SVM).")
    ap.add_argument("--data", help="CSV/TSV of labelled emails (see module docstring)")
    ap.add_argument("--samples", type=int, default=2000, help="size of the built-in sample data")
    ap.add_argument("--model-path", default="spam_model.joblib", help="where the trained model is saved/loaded")
    ap.add_argument("--predict", action="append", metavar="TEXT", help="classify this email text (repeatable)")
    ap.add_argument("--predict-file", nargs="+", metavar="FILE", help="classify the contents of these text files")
    ap.add_argument("--save-sample", metavar="DIR", help="write the sample data to DIR/emails.csv and exit")
    args = ap.parse_args(argv)

    pd.set_option("display.width", 200, "display.max_colwidth", 80)

    if args.save_sample:
        out = Path(args.save_sample)
        out.mkdir(parents=True, exist_ok=True)
        df = build_sample_data(args.samples)
        df.assign(label=df["label"].map({1: "spam", 0: "ham"})).to_csv(out / "emails.csv", index=False)
        print(f"wrote {out / 'emails.csv'} ({len(df)} emails)")
        return

    # ---- filter mode: use a saved model ----
    if args.predict or args.predict_file:
        if not Path(args.model_path).exists():
            sys.exit(f"error: no model at {args.model_path!r}; run without --predict first to train one")
        saved = joblib.load(args.model_path)
        texts = list(args.predict or [])
        texts = [t.encode().decode("unicode_escape") for t in texts]  # allow \n typed on the command line
        for f in args.predict_file or []:
            texts.append(Path(f).read_text(encoding="utf-8", errors="replace"))
        print(f"Model: {saved['name']}  (score: " +
              ("probability of spam" if hasattr(saved["model"], "predict_proba") else "SVM margin, >0 = spam") + ")")
        print(classify(saved["model"], texts).to_string(index=False))
        return

    # ---- training mode ----
    try:
        df = load_dataset(args.data) if args.data else build_sample_data(args.samples)
    except (OSError, ValueError) as e:
        sys.exit(f"error: {e}")
    n_spam = int(df["label"].sum())
    print(f"Data: {len(df)} emails, {n_spam} spam ({100 * n_spam / len(df):.0f}%), {len(df) - n_spam} ham"
          + ("" if args.data else "  [built-in SYNTHETIC sample]"))

    res, searches, (n_tr, n_te) = train_and_compare(df)
    print(f"\nTrained on {n_tr}, tested on {n_te} held-out emails. Spam is the positive class.")
    print("FP = legitimate mail sent to spam (the costly mistake), FN = spam that got through.\n")
    print(res.round(3).to_string())

    for name, gs in searches.items():
        spam_t, ham_t = top_terms(gs.best_estimator_)
        print(f"\n{name} - strongest spam terms: {', '.join(spam_t)}")
        print(f"{' ' * len(name)}   strongest ham terms:  {', '.join(ham_t)}")

    best = res["cv F1"].idxmax()  # chosen on training-set CV, not on the test set
    final = clone(searches[best].best_estimator_).fit(df["text"], df["label"])
    joblib.dump({"name": best, "model": final}, args.model_path)
    print(f"\nBest by cross-validated F1: {best}. Refit on all data and saved to {args.model_path}")

    print("\nSanity check on six hand-written emails (expected: ham, SPAM, SPAM, ham, ham, SPAM):")
    print(classify(final, DEMO_EMAILS).to_string(index=False))

    if not args.data:
        print("\nNote: the sample data is synthetic and template-based, so these scores are far higher than a "
              "real-world spam filter would get. Use --data with a real corpus for meaningful numbers.")


if __name__ == "__main__":
    # Pass an empty list to main to avoid parsing Colab's internal runtime arguments
    main([])


Data: 2000 emails, 613 spam (31%), 1387 ham  [built-in SYNTHETIC sample]

Trained on 1600, tested on 400 held-out emails. Spam is the positive class.
FP = legitimate mail sent to spam (the costly mistake), FN = spam that got through.

                 best params  cv F1  accuracy  precision  recall     F1  ham->spam (FP)  spam missed (FN)
model                                                                                                    
Naive Bayes  {'alpha': 0.01}   0.97     0.985      0.976   0.976  0.976               3                 3
Linear SVM        {'C': 0.1}   0.97     0.985      0.976   0.976  0.976               3                 3

Naive Bayes - strongest spam terms: home, boss, earn, earn moneytoken, click, suspended, account suspended, working home
              strongest ham terms:  statement, meeting, password email, password, view, available, ask, app urltoken

Linear SVM - strongest spam terms: today, moneytoken, needed, claim, overdue, win, win moneytoken, se